# Functional and Non-Functional Metrics 

Let's take a look at both functional & non-functional metrics: 

## Functional Metrics 

**Functional metrics** assess directly how suitable the model is for the targeted use cases. It assess how well the model performs in terms of accuracy, relevance, and task completion. So, they directly measure the ability to generate correct and useful outputs.

📌 For function calling, this means:
- Does the model predict the correct function?
- Does it include the right parameters?
- Does it avoid hallucinating extra parameters?

Consider the following example of an **expected model** response (ie. ground truth): 

> ```json
> {
>   "function_name": "set_volume",
>   "parameters": {
>     "volume_level": 22
>   }
> }
> ```

There are multiple different outcomes which could occur as a result of a prediction...

### The happy path 😊 

#### ✅ Model correctly predicted the function and parameters (`passed`)

```jsonc
{
  "function_name": "set_volume",
  "parameters": {
    "volume_level": 22
  }
}

```

So far, so good. What could possibly go wrong? 

**🤔 Question:** How many types of errors can you think of? Try writing them down at least 5 before proceeding. 

<br>

### The less-than-perfect path ... 🤷‍♂️

#### ❌ Incorrect Predictions

Failed: Any of the following scenarios:
- Any of the parameters is incorrect
- The function is incorrect
- No function was returned
- Did not return a valid json

```jsonc
{
  "function_name": "set_volume",
  "parameters": {
    "volume_level": 10 // Failed: The `volume_level` parameter is incorrect, should be 22 instead
  }
}

{
  "function_name": "set_volumes", // Failed: The function is incorrect
  "parameters": {
    "volume_level": 22
  }
}

{ // Failed: no function was returned
  "parameters": {
    "volume_level": 22
  }
}

// Failed: Invalid response format
Sure, here is the answer: {
  "function_name": "set_volume",
  "parameters": {
    "volume_level": 22
  },
}

{ // Failed: Missing argument
  "function_name": "set_volume", 
  "parameters": {
    "volume_level": ,
  }
}
```

**🤔 Questions:** Should we be interested only in whether the output was "correct or incorrect" (black or white) - ie. two metrics ?  Or should we create additional metrics to measure the levels of correctness?  If so, what would you measure? 



### Defining Detailed Functional Metrics

We can define some additional metrics to help pinpoint **where** the model makes mistakes.

| Metric Name             | Definition          | 
|-------------------------|---------------------|
| passed	                | Did the model return the exact expected function? (1 = Yes, 0 = No) | 
| correct_function_called	| Did the model select the correct function?                     |
| function_exists	        | Does the function exist in the predefined specification?       |
| function_name_specified	| Did the model specify a function name?                         |
| correctly_generated_arguments	| Precision: Correct arguments / Total predicted arguments |
| correctly_retrieved_arguments	| Recall: Correct arguments / Total expected arguments     |
| extra_argument	              | Number of unexpected (hallucinated) arguments            |


Here are some examples: 

##### The correct function was called (`correct_function_called`)

```jsonc
{
  "function_name": "set_volume", // Correct function called
  "parameters": {
    "volume_level": "foo"
  }
}
```

##### The function exists or not in the function specification list (`function_exists`)

```jsonc
{
  "function_name": "get_navigation_directions", // The function exists, even though it's incorrect
  "parameters": {
    "volume_level": "foo"
  }
}
```

##### No function was called (`function_name_specified`)

```jsonc
{
  // No function was called here
  "parameters": {
    "volume_level": "foo"
  }
}
```

##### Proportion of correct argument value retrieved over the number of retrieved parameters (`correctly_generated_arguments` - precision)

```jsonc
// 1
{
  "function_name": "set_volume",
  "parameters": {
    "volume_level": 22
  }
}

// 0.5 - out of the 2 predicted arguments, only one is correct
{
  "function_name": "set_volume",
  "parameters": {
    "volume_level": 22,
    "set": "stereo"
  }
}

// 0 
{
  "function_name": "set_volume",
  "parameters": {
    "volume_level": 10
  }
}
```

#####  Proportion of retrieved argument values retrieved over the total of expected (ground truth) parameters (`correctly_retrieved_arguments` - recall)

```jsonc
// 0
{
  "function_name": "set_volume",
  "parameters": {
    // Missing the `volume_level` argument
  }
}

// 0
{
  "function_name": "set_volume",
  "parameters": {
    "set": "stereo" // The `set` argument does not exist in the function specification, so should be ignored
    // Missing the `volume_level` argument
  }
}

// 1 - The extra "set" argument will be captured in the extra argument metric
{
  "function_name": "set_volume",
  "parameters": {
    "set": "stereo",
    "volume_level": 22
  }
}
```

#####  Extra argument (`extra_argument` - including hallucination)

```jsonc
{
  "function_name": "set_volume",
  "parameters": {
    "volume_level": 22,
    "category": "stereo" // The `category` argument should not be specified
  }
}

{
  "function_name": "set_volume",
  "parameters": {
    "volume_level": 22,
    "set": null // the `set` argument does not exist in the function specification
  }
}

{
  "function_name": "set_volume",
  "parameters": {
    "volume_level": 22,
    "set": "stereo" // the `set` argument does not exist in the function specification
  }
}

```


## Non-functional metrics

**Non-functional metrics** focus on the overall reliability and usability of the model. It measures how well the model operates in terms of speed, cost, reliability, and ethical concerns rather than task performance.  

Common metrics for language models include: 

- **Time to first token** (`secs_to_first_token`): This tells us how long does the model spend thinking. For the function calling task, it is not as important as the time to last token, but it is still useful to know. In the future, this will be useful in understanding the user experience for cases where we return text answers, such as clarifying questions, acknowledgement, etc.

- **Time to last token** (`secs_to_last_token`): This is crucial, because function calls can only make sense when the function specification is fully populated

- **Tokens per second** (`tokens_per_second`): Measure the generation speed. Depending on the setup, and the complexity of the function call, the token generation step might take up a large chunk of the latency, so this is crucial to measure. 


**🤔 Challenge: can you think of at least 5 other non-functional metrics for SLMs?** 

<details>
    <summary>Click for answer</summary>

1. Memory footprint - the amount of RAM & GPU memory usage during inference.

2. Compute cost -  financial cost per function call. 

3. Robustness to adversarial inputs - how well the model handles unexpected, misleading, or malicious inputs. Useful for improving security and preventing prompt injection attacks.

4. Scalability - performance under heavy concurrent requests

5. Energy consumption -  Helps optimize for [sustainable AI](https://www.datacamp.com/blog/sustainable-ai).

</details>